# Bearing Autoencoder — PER ASSET EXPERIMENT

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
from collections import deque

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from sklearn.svm import OneClassSVM
from sklearn.covariance import EllipticEnvelope

from sklearn.pipeline import Pipeline

import pandas as pd
import numpy as np

In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model

GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)
tf.random.set_seed(GLOBAL_SEED)
print('TF version:', tf.__version__)

TF version: 2.16.2


In [3]:
data_root = Path("../CWRU_Bearing_NumPy-main/Data")

In [4]:
def load_cwru_signal(bearing_id, regime):
    bearing_rpm = f"{bearing_id} RPM"
    folder = data_root / bearing_rpm

    if regime == "healthy":
        d = np.load(folder / f"{bearing_id}_Normal.npz")
        return d["DE"]

    patterns = {
        "ball":  f"{bearing_id}_B_*_DE12.npz",
        "inner": f"{bearing_id}_IR_*_DE12.npz",
        "outer": f"{bearing_id}_OR*@*_DE12.npz",
    }

    signals = []
    for f in sorted(folder.glob(patterns[regime])):
        d = np.load(f)
        signals.append(d["DE"])

    return np.concatenate(signals)

In [5]:
def make_windows(x, win=1200, step=1200):
    n = (len(x) - win) // step + 1
    return np.stack([x[i*step:i*step+win] for i in range(n)], axis=0)

# ── Compute CWRU MSEs first ─────────────────────────────────────────────
def cwru_mse(signal_1d):
    ws    = make_windows(signal_1d, win=1200, step=1200)
    ws_2d = ws.reshape(ws.shape[0], -1)
    ws_sc = scaler.transform(ws_2d)
    X     = ws_sc[:, :, np.newaxis].astype('float32')
    X_rec = ae.predict(X, verbose=0)
    return np.mean((X - X_rec)**2, axis=(1, 2))

In [6]:
BEARING_IDS = ("1730", "1750", "1772", "1797")
WIN = 1200
STEP = 1200
THRESHOLD_PERCENTILE = 98.0


def load_asset_windows(
    bearing_id: str,
    *,
    win: int = WIN,
    step: int = STEP,
):
    """
    Load and window CWRU exactly as in bearing_autoencoder.ipynb.

    No train/calibration/test split is introduced. This is the ideal,
    asset-specific oracle: all available healthy observations are used
    for model fitting and threshold calibration.
    """
    signals = {
        "healthy": load_cwru_signal(bearing_id, "healthy"),
        "ball": load_cwru_signal(bearing_id, "ball"),
        "inner_race": load_cwru_signal(bearing_id, "inner"),
        "outer_race": load_cwru_signal(bearing_id, "outer"),
    }

    return {
        regime: make_windows(
            signal,
            win=win,
            step=step,
        ).astype(np.float32)
        for regime, signal in signals.items()
    }


def evaluate_scores(
    *,
    bearing_id: str,
    detector_name: str,
    healthy_scores: np.ndarray,
    ball_scores: np.ndarray,
    inner_scores: np.ndarray,
    outer_scores: np.ndarray,
    threshold_percentile: float = THRESHOLD_PERCENTILE,
):
    """
    Common evaluation for all detectors.

    Larger score must always mean more anomalous.
    """
    tau = float(
        np.percentile(
            healthy_scores,
            threshold_percentile,
        )
    )

    return {
        "bearing_id": bearing_id,
        "detector": detector_name,
        "threshold_percentile": threshold_percentile,
        "threshold": tau,
        "healthy_far": 100.0 * np.mean(healthy_scores > tau),
        "ball_dr": 100.0 * np.mean(ball_scores > tau),
        "inner_race_dr": 100.0 * np.mean(inner_scores > tau),
        "outer_race_dr": 100.0 * np.mean(outer_scores > tau),
        "n_healthy": len(healthy_scores),
        "n_ball": len(ball_scores),
        "n_inner_race": len(inner_scores),
        "n_outer_race": len(outer_scores),
    }

#### TEST ONE-CLASS SVM

In [7]:
ocsvm_results = []

for bearing_id in BEARING_IDS:
    print(f"Running asset-specific OC-SVM for {bearing_id} RPM...")

    windows = load_asset_windows(bearing_id)

    # Asset-specific preprocessing fitted only on its real healthy data.
    scaler = StandardScaler()
    X_healthy_raw = windows["healthy"].reshape(len(windows["healthy"]), -1)
    X_ball_raw = windows["ball"].reshape(len(windows["ball"]), -1)
    X_inner_raw = windows["inner_race"].reshape(len(windows["inner_race"]), -1)
    X_outer_raw = windows["outer_race"].reshape(len(windows["outer_race"]), -1)

    scaler = StandardScaler()

    X_healthy = scaler.fit_transform(X_healthy_raw)
    X_ball = scaler.transform(X_ball_raw)
    X_inner = scaler.transform(X_inner_raw)
    X_outer = scaler.transform(X_outer_raw)

    # PCA is trained only on this asset's healthy observations.
    # It prevents the RBF OC-SVM from operating directly in 1200 dimensions.
    n_components = min(
        32,
        X_healthy.shape[0] - 1,
        X_healthy.shape[1],
    )

    detector = Pipeline(
        [
            (
                "pca",
                PCA(
                    n_components=n_components,
                    random_state=GLOBAL_SEED,
                ),
            ),
            (
                "ocsvm",
                OneClassSVM(
                    kernel="rbf",
                    nu=0.02,
                    gamma="scale",
                ),
            ),
        ]
    )

    detector.fit(X_healthy)

    # sklearn returns larger score_samples for more nominal samples.
    # Negate so that larger always means more anomalous.
    healthy_scores = -detector.score_samples(X_healthy)
    ball_scores = -detector.score_samples(X_ball)
    inner_scores = -detector.score_samples(X_inner)
    outer_scores = -detector.score_samples(X_outer)

    ocsvm_results.append(
        evaluate_scores(
            bearing_id=bearing_id,
            detector_name="Asset-specific OC-SVM",
            healthy_scores=healthy_scores,
            ball_scores=ball_scores,
            inner_scores=inner_scores,
            outer_scores=outer_scores,
        )
    )

ocsvm_results = pd.DataFrame(ocsvm_results)
display(ocsvm_results)

Running asset-specific OC-SVM for 1730 RPM...
Running asset-specific OC-SVM for 1750 RPM...
Running asset-specific OC-SVM for 1772 RPM...
Running asset-specific OC-SVM for 1797 RPM...


,bearing_id,detector,threshold_percentile,threshold,healthy_far,ball_dr,inner_race_dr,outer_race_dr,n_healthy,n_ball,n_inner_race,n_outer_race
0,1730,Asset-specific OC-SVM,98.0,-0.999212,2.227723,0.000000,25.123153,4.915730,404,405,406,712
1,1750,Asset-specific OC-SVM,98.0,-1.069604,2.227723,12.345679,25.185185,5.907173,404,405,405,711
2,1772,Asset-specific OC-SVM,98.0,-0.953015,2.233251,0.000000,24.938272,0.000000,403,405,405,712
3,1797,Asset-specific OC-SVM,98.0,-0.604553,2.463054,0.000000,24.938272,0.000000,203,406,405,711


#### TEST ELLIPTICAL ENVELOPE

In [8]:
elliptic_results = []

for bearing_id in BEARING_IDS:
    print(f"Running asset-specific Elliptic Envelope for {bearing_id} RPM...")

    windows = load_asset_windows(bearing_id)

    # Flatten exactly as for OC-SVM
    X_healthy_raw = windows["healthy"].reshape(len(windows["healthy"]), -1)
    X_ball_raw = windows["ball"].reshape(len(windows["ball"]), -1)
    X_inner_raw = windows["inner_race"].reshape(len(windows["inner_race"]), -1)
    X_outer_raw = windows["outer_race"].reshape(len(windows["outer_race"]), -1)

    # Asset-specific preprocessing fitted only on real healthy data
    scaler = StandardScaler()

    X_healthy = scaler.fit_transform(X_healthy_raw)
    X_ball = scaler.transform(X_ball_raw)
    X_inner = scaler.transform(X_inner_raw)
    X_outer = scaler.transform(X_outer_raw)

    # Robust covariance estimation is unstable in 1200-D with only a few
    # hundred healthy windows, so reduce dimensionality using healthy-only PCA.
    n_components = min(
        16,
        X_healthy.shape[0] - 1,
        X_healthy.shape[1],
    )

    detector = Pipeline(
        [
            (
                "pca",
                PCA(
                    n_components=n_components,
                    random_state=GLOBAL_SEED,
                ),
            ),
            (
                "elliptic",
                EllipticEnvelope(
                    contamination=0.02,
                    support_fraction=None,
                    random_state=GLOBAL_SEED,
                ),
            ),
        ]
    )

    detector.fit(X_healthy)

    # sklearn score_samples:
    # larger = more nominal
    # negate so larger = more anomalous
    healthy_scores = -detector.score_samples(X_healthy)
    ball_scores = -detector.score_samples(X_ball)
    inner_scores = -detector.score_samples(X_inner)
    outer_scores = -detector.score_samples(X_outer)

    elliptic_results.append(
        evaluate_scores(
            bearing_id=bearing_id,
            detector_name="Asset-specific Elliptic Envelope",
            healthy_scores=healthy_scores,
            ball_scores=ball_scores,
            inner_scores=inner_scores,
            outer_scores=outer_scores,
        )
    )

elliptic_results = pd.DataFrame(elliptic_results)
display(elliptic_results)

Running asset-specific Elliptic Envelope for 1730 RPM...
Running asset-specific Elliptic Envelope for 1750 RPM...
Running asset-specific Elliptic Envelope for 1772 RPM...
Running asset-specific Elliptic Envelope for 1797 RPM...


,bearing_id,detector,threshold_percentile,threshold,healthy_far,ball_dr,inner_race_dr,outer_race_dr,n_healthy,n_ball,n_inner_race,n_outer_race
0,1730,Asset-specific Elliptic Envelope,98.0,26.031561,2.227723,0.000000,25.369458,6.460674,404,405,406,712
1,1750,Asset-specific Elliptic Envelope,98.0,32.522446,2.227723,2.469136,17.777778,0.562588,404,405,405,711
2,1772,Asset-specific Elliptic Envelope,98.0,39.481334,2.233251,0.000000,0.000000,0.000000,403,405,405,712
3,1797,Asset-specific Elliptic Envelope,98.0,68.972761,2.463054,0.000000,14.567901,0.000000,203,406,405,711


In [9]:
LATENT_DIM = 16
SIG_LEN = 1200

def build_autoencoder(sig_len, latent_dim):
    # ── Encoder ──────────────────────────────────────────────────────────
    inp = keras.Input(shape=(sig_len, 1), name='signal_in')
    x = layers.Conv1D(32,  kernel_size=16, strides=2, padding='same', activation='relu')(inp)
    x = layers.Conv1D(64,  kernel_size=8,  strides=2, padding='same', activation='relu')(x)
    x = layers.Conv1D(128, kernel_size=4,  strides=2, padding='same', activation='relu')(x)
    conv_shape = x.shape[1:]            # remember shape before flatten
    x = layers.Flatten()(x)
    latent = layers.Dense(
        latent_dim,
        activity_regularizer=keras.regularizers.l1(1.5e-4),
        name='latent'
    )(x)


    # ── Decoder ──────────────────────────────────────────────────────────
    y = layers.Dense(conv_shape[0] * conv_shape[1], activation='relu')(latent)
    y = layers.Reshape(conv_shape)(y)
    y = layers.Conv1DTranspose(128, kernel_size=4,  strides=2, padding='same', activation='relu')(y)
    y = layers.Conv1DTranspose(64,  kernel_size=8,  strides=2, padding='same', activation='relu')(y)
    y = layers.Conv1DTranspose(32,  kernel_size=16, strides=2, padding='same', activation='relu')(y)
    # Final layer: crop or pad to exact sig_len, linear activation
    y = layers.Conv1D(1, kernel_size=1, padding='same', activation='linear', name='signal_out')(y)
    y = layers.Cropping1D((0, y.shape[1] - sig_len))(y) if y.shape[1] > sig_len else y

    autoencoder = Model(inp, y, name='bearing_autoencoder')
    encoder     = Model(inp, latent, name='encoder')
    return autoencoder, encoder

ae, encoder = build_autoencoder(SIG_LEN, LATENT_DIM)
ae.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')
ae.summary()

2026-07-16 17:34:09.373271: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2 Pro
2026-07-16 17:34:09.373314: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-07-16 17:34:09.373334: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-07-16 17:34:09.373361: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-07-16 17:34:09.373381: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "bearing_autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ signal_in (InputLayer)          │ (None, 1200, 1)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 600, 32)        │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 300, 64)        │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 150, 128)       │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 19200)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ latent (Dense)                  │ (None, 16)             │       307,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 19200)          │       326,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 150, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_transpose                │ (None, 300, 128)       │        65,664 │
│ (Conv1DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_transpose_1              │ (None, 600, 64)        │        65,600 │
│ (Conv1DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_transpose_2              │ (None, 1200, 32)       │        32,800 │
│ (Conv1DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ signal_out (Conv1D)             │ (None, 1200, 1)        │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 847,601 (3.23 MB)

 Trainable params: 847,601 (3.23 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
STABILITY_SEEDS = [7, 21, 42, 84, 126]

ae_results = []

for seed in STABILITY_SEEDS:

    print(f"\n==========================")
    print(f"Seed {seed}")
    print(f"==========================")

    keras.backend.clear_session()

    np.random.seed(seed)
    tf.random.set_seed(seed)

    try:
        keras.utils.set_random_seed(seed)
    except Exception:
        pass

    for bearing_id in BEARING_IDS:

        print(f"Training asset-specific AE for {bearing_id} RPM...")

        windows = load_asset_windows(bearing_id)

        # -------------------------------------------------------
        # preprocessing
        # -------------------------------------------------------

        X_healthy_raw = windows["healthy"].reshape(len(windows["healthy"]), -1)
        X_ball_raw = windows["ball"].reshape(len(windows["ball"]), -1)
        X_inner_raw = windows["inner_race"].reshape(len(windows["inner_race"]), -1)
        X_outer_raw = windows["outer_race"].reshape(len(windows["outer_race"]), -1)

        scaler = StandardScaler()

        X_healthy_sc = scaler.fit_transform(X_healthy_raw).astype(np.float32)
        X_ball_sc = scaler.transform(X_ball_raw).astype(np.float32)
        X_inner_sc = scaler.transform(X_inner_raw).astype(np.float32)
        X_outer_sc = scaler.transform(X_outer_raw).astype(np.float32)

        X_healthy = X_healthy_sc[..., np.newaxis]
        X_ball = X_ball_sc[..., np.newaxis]
        X_inner = X_inner_sc[..., np.newaxis]
        X_outer = X_outer_sc[..., np.newaxis]

        # -------------------------------------------------------
        # model
        # -------------------------------------------------------

        ae, encoder = build_autoencoder(
            sig_len=X_healthy.shape[1],
            latent_dim=LATENT_DIM,
        )

        ae.compile(
            optimizer=keras.optimizers.Adam(1e-3),
            loss="mse",
        )

        callbacks = [
            keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=10,
                restore_best_weights=True,
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                factor=0.5,
                patience=5,
                min_lr=1e-5,
            ),
        ]

        ae.fit(
            X_healthy,
            X_healthy,
            validation_split=0.20,
            epochs=100,
            batch_size=32,
            shuffle=True,
            callbacks=callbacks,
            verbose=0,
        )

        # -------------------------------------------------------
        # scores
        # -------------------------------------------------------

        healthy_rec = ae.predict(X_healthy, verbose=0)
        ball_rec = ae.predict(X_ball, verbose=0)
        inner_rec = ae.predict(X_inner, verbose=0)
        outer_rec = ae.predict(X_outer, verbose=0)

        healthy_scores = np.mean((X_healthy - healthy_rec) ** 2, axis=(1, 2))
        ball_scores    = np.mean((X_ball    - ball_rec)    ** 2, axis=(1, 2))
        inner_scores   = np.mean((X_inner   - inner_rec)   ** 2, axis=(1, 2))
        outer_scores   = np.mean((X_outer   - outer_rec)   ** 2, axis=(1, 2))

        row = evaluate_scores(
            bearing_id=bearing_id,
            detector_name="Asset-specific AE",
            healthy_scores=healthy_scores,
            ball_scores=ball_scores,
            inner_scores=inner_scores,
            outer_scores=outer_scores,
        )

        row["seed"] = seed

        ae_results.append(row)

# -------------------------------------------------------
# final dataframe
# -------------------------------------------------------

ae_results = pd.DataFrame(ae_results)

display(ae_results)


Seed 7
Training asset-specific AE for 1730 RPM...


2026-07-16 17:34:10.529814: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


Training asset-specific AE for 1750 RPM...
Training asset-specific AE for 1772 RPM...
Training asset-specific AE for 1797 RPM...

Seed 21
Training asset-specific AE for 1730 RPM...
Training asset-specific AE for 1750 RPM...
Training asset-specific AE for 1772 RPM...
Training asset-specific AE for 1797 RPM...

Seed 42
Training asset-specific AE for 1730 RPM...
Training asset-specific AE for 1750 RPM...
Training asset-specific AE for 1772 RPM...
Training asset-specific AE for 1797 RPM...

Seed 84
Training asset-specific AE for 1730 RPM...
Training asset-specific AE for 1750 RPM...
Training asset-specific AE for 1772 RPM...
Training asset-specific AE for 1797 RPM...

Seed 126
Training asset-specific AE for 1730 RPM...
Training asset-specific AE for 1750 RPM...
Training asset-specific AE for 1772 RPM...
Training asset-specific AE for 1797 RPM...


,bearing_id,detector,threshold_percentile,threshold,healthy_far,ball_dr,inner_race_dr,outer_race_dr,n_healthy,n_ball,n_inner_race,n_outer_race,seed
0,1730,Asset-specific AE,98.0,0.658501,2.227723,100.0,100.0,100.0,404,405,406,712,7
1,1750,Asset-specific AE,98.0,0.666244,2.227723,100.0,100.0,100.0,404,405,405,711,7
2,1772,Asset-specific AE,98.0,0.883316,2.233251,100.0,100.0,100.0,403,405,405,712,7
3,1797,Asset-specific AE,98.0,0.446139,2.463054,100.0,100.0,100.0,203,406,405,711,7
4,1730,Asset-specific AE,98.0,0.672507,2.227723,100.0,100.0,100.0,404,405,406,712,21
5,1750,Asset-specific AE,98.0,1.164796,2.227723,100.0,100.0,100.0,404,405,405,711,21
6,1772,Asset-specific AE,98.0,0.610576,2.233251,100.0,100.0,100.0,403,405,405,712,21
7,1797,Asset-specific AE,98.0,0.473301,2.463054,100.0,100.0,100.0,203,406,405,711,21
8,1730,Asset-specific AE,98.0,0.619351,2.227723,100.0,100.0,100.0,404,405,406,712,42
9,1750,Asset-specific AE,98.0,0.656872,2.227723,100.0,100.0,100.0,404,405,405,711,42


In [11]:
stability = (
    ae_results
    .groupby("bearing_id")
    .agg(
        healthy_far_mean=("healthy_far", "mean"),
        healthy_far_std=("healthy_far", "std"),
        ball_dr_mean=("ball_dr", "mean"),
        ball_dr_std=("ball_dr", "std"),
        inner_race_dr_mean=("inner_race_dr", "mean"),
        inner_race_dr_std=("inner_race_dr", "std"),
        outer_race_dr_mean=("outer_race_dr", "mean"),
        outer_race_dr_std=("outer_race_dr", "std"),
        threshold_mean=("threshold", "mean"),
        threshold_std=("threshold", "std"),
    )
    .reset_index()
)

display(stability)

,bearing_id,healthy_far_mean,healthy_far_std,ball_dr_mean,ball_dr_std,inner_race_dr_mean,inner_race_dr_std,outer_race_dr_mean,outer_race_dr_std,threshold_mean,threshold_std
0,1730,2.227723,0.0,100.0,0.0,100.0,0.0,100.0,0.0,0.648529,0.020741
1,1750,2.227723,0.0,100.0,0.0,100.0,0.0,100.0,0.0,0.759031,0.226892
2,1772,2.233251,0.0,100.0,0.0,100.0,0.0,100.0,0.0,0.724833,0.148432
3,1797,2.463054,0.0,100.0,0.0,100.0,0.0,100.0,0.0,0.459058,0.026359


### METHOD TEST

In [ ]:
# =====================================================================
# TRAIN THE FIXED OEM / SHARED SIMULATOR-TRAINED MODEL
# =====================================================================

from bearing_simulator import BearingSignalSimulator

# ---------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------

OEM_SEED = 42

keras.backend.clear_session()
keras.utils.set_random_seed(OEM_SEED)
np.random.seed(OEM_SEED)
tf.random.set_seed(OEM_SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass


# ---------------------------------------------------------------------
# Synthetic dataset configuration
# ---------------------------------------------------------------------

shared_simulator = BearingSignalSimulator()
shared_sig_len = len(shared_simulator.t)

SYNTHETIC_REGIMES = [
    "healthy",
    "outer_race",
    "inner_race",
    "ball_fault",
]

SYNTHETIC_LABELS = {
    "healthy": 0,
    "outer_race": 1,
    "inner_race": 2,
    "ball_fault": 3,
}

N_SYNTHETIC_TRAIN = 4_000
N_SYNTHETIC_VAL = 1_000
N_SYNTHETIC_TEST = 750

SHARED_LATENT_DIM = 16
SHARED_THRESHOLD_PERCENTILE = 98.0

print(f"Shared synthetic signal length: {shared_sig_len} samples")


# ---------------------------------------------------------------------
# Generate independent healthy training and validation datasets
# ---------------------------------------------------------------------

X_shared_train_raw = np.asarray(
    [
        shared_simulator.sample("healthy", mc=True)
        for _ in range(N_SYNTHETIC_TRAIN)
    ],
    dtype=np.float32,
)

X_shared_val_raw = np.asarray(
    [
        shared_simulator.sample("healthy", mc=True)
        for _ in range(N_SYNTHETIC_VAL)
    ],
    dtype=np.float32,
)


# ---------------------------------------------------------------------
# Fit preprocessing only on synthetic healthy training data
# ---------------------------------------------------------------------

shared_scaler = StandardScaler()

X_shared_train_sc = shared_scaler.fit_transform(
    X_shared_train_raw
).astype(np.float32)

X_shared_val_sc = shared_scaler.transform(
    X_shared_val_raw
).astype(np.float32)

X_shared_train = X_shared_train_sc[..., np.newaxis]
X_shared_val = X_shared_val_sc[..., np.newaxis]


# ---------------------------------------------------------------------
# Generate a separate synthetic test set
# ---------------------------------------------------------------------

shared_test_by_regime = {}
shared_y_test_parts = []

for regime in SYNTHETIC_REGIMES:
    regime_raw = np.asarray(
        [
            shared_simulator.sample(regime, mc=True)
            for _ in range(N_SYNTHETIC_TEST)
        ],
        dtype=np.float32,
    )

    regime_sc = shared_scaler.transform(
        regime_raw
    ).astype(np.float32)

    shared_test_by_regime[regime] = regime_sc[..., np.newaxis]

    shared_y_test_parts.append(
        np.full(
            N_SYNTHETIC_TEST,
            SYNTHETIC_LABELS[regime],
            dtype=np.int64,
        )
    )

X_shared_test = np.concatenate(
    [
        shared_test_by_regime[regime]
        for regime in SYNTHETIC_REGIMES
    ],
    axis=0,
)

y_shared_test = np.concatenate(
    shared_y_test_parts,
    axis=0,
)

print(
    f"Train: {X_shared_train.shape} | "
    f"Validation: {X_shared_val.shape} | "
    f"Test: {X_shared_test.shape}"
)


# ---------------------------------------------------------------------
# Build and train the fixed shared/OEM autoencoder
# ---------------------------------------------------------------------

shared_ae, shared_encoder = build_autoencoder(
    sig_len=shared_sig_len,
    latent_dim=SHARED_LATENT_DIM,
)

shared_ae.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse",
)

shared_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
    ),
]

shared_history = shared_ae.fit(
    X_shared_train,
    X_shared_train,
    validation_data=(X_shared_val, X_shared_val),
    epochs=100,
    batch_size=32,
    shuffle=True,
    callbacks=shared_callbacks,
    verbose=1,
)


# ---------------------------------------------------------------------
# Calibrate the simulator threshold on held-out healthy validation data
# ---------------------------------------------------------------------

X_shared_val_rec = shared_ae.predict(
    X_shared_val,
    verbose=0,
)

shared_val_mse = np.mean(
    np.square(X_shared_val - X_shared_val_rec),
    axis=(1, 2),
)

shared_tau_mc = float(
    np.percentile(
        shared_val_mse,
        SHARED_THRESHOLD_PERCENTILE,
    )
)

print(
    f"\nSynthetic validation threshold "
    f"τ_MC (p{SHARED_THRESHOLD_PERCENTILE:.0f}) "
    f"= {shared_tau_mc:.6f}"
)


# ---------------------------------------------------------------------
# Evaluate on the independent synthetic test set
# ---------------------------------------------------------------------

X_shared_test_rec = shared_ae.predict(
    X_shared_test,
    verbose=0,
)

shared_test_mse = np.mean(
    np.square(X_shared_test - X_shared_test_rec),
    axis=(1, 2),
)


def _synthetic_rate(regime: str) -> float:
    regime_mask = (
        y_shared_test == SYNTHETIC_LABELS[regime]
    )

    return 100.0 * float(
        np.mean(
            shared_test_mse[regime_mask] > shared_tau_mc
        )
    )


synthetic_ae_results = pd.DataFrame(
    [
        {
            "bearing_id": "Synthetic",
            "detector": "Shared simulator-trained AE",
            "threshold_percentile": SHARED_THRESHOLD_PERCENTILE,
            "threshold": shared_tau_mc,
            "healthy_far": _synthetic_rate("healthy"),
            "ball_dr": _synthetic_rate("ball_fault"),
            "inner_race_dr": _synthetic_rate("inner_race"),
            "outer_race_dr": _synthetic_rate("outer_race"),
            "n_healthy": N_SYNTHETIC_TEST,
            "n_ball": N_SYNTHETIC_TEST,
            "n_inner_race": N_SYNTHETIC_TEST,
            "n_outer_race": N_SYNTHETIC_TEST,
            "training_seed": OEM_SEED,
            "n_train": N_SYNTHETIC_TRAIN,
            "n_validation": N_SYNTHETIC_VAL,
        }
    ]
)

display(synthetic_ae_results)


# ---------------------------------------------------------------------
# Sanity checks
# ---------------------------------------------------------------------

assert shared_ae is not None
assert shared_scaler is not None
assert shared_sig_len == X_shared_train.shape[1]
assert np.isfinite(shared_tau_mc)

print("\nFixed OEM artifacts created:")
print("  shared_ae")
print("  shared_encoder")
print("  shared_scaler")
print("  shared_tau_mc")
print("  shared_history")

Shared synthetic signal length: 1200 samples
Train: (4000, 1200, 1) | Validation: (1000, 1200, 1) | Test: (3000, 1200, 1)
Epoch 1/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 35ms/step - loss: 0.2818 - val_loss: 0.0914 - learning_rate: 0.0010
Epoch 2/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - loss: 0.0780 - val_loss: 0.0711 - learning_rate: 0.0010
Epoch 3/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 32ms/step - loss: 0.0690 - val_loss: 0.0702 - learning_rate: 0.0010
Epoch 4/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - loss: 0.0699 - val_loss: 0.0685 - learning_rate: 0.0010
Epoch 5/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - loss: 0.0674 - val_loss: 0.0673 - learning_rate: 0.0010
Epoch 6/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - loss: 0.0660 - val_loss: 0.0663 - learning_rate: 0.0010
Epoch 7/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - loss: 0.0670 - val_loss: 0.0722 - learning_rate: 0.0010
Epoch 8/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - loss: 0.0667 - val_loss: 0.0673 - lear

,bearing_id,detector,threshold_percentile,threshold,healthy_far,ball_dr,inner_race_dr,outer_race_dr,n_healthy,n_ball,n_inner_race,n_outer_race,training_seed,n_train,n_validation
0,Synthetic,Shared simulator-trained AE,98.0,0.103207,1.2,95.6,100.0,100.0,750,750,750,750,42,4000,1000



Fixed OEM artifacts created:
  shared_ae
  shared_encoder
  shared_scaler
  shared_tau_mc
  shared_history


### DOMAIN ADAPTATION ###

In [20]:
WINDOW_SIZE = 100
# =====================================================================
# Online operational-radius estimator
# =====================================================================

class AdaptiveOperationalRadius:
    """
    Online empirical-quantile estimator.

    The globally shared quantity is the target nominal coverage
    (e.g. 98%), not a z-score or an absolute threshold.

    The local operational radius is estimated as the empirical
    percentile of the accepted nominal residuals.
    """

    def __init__(
        self,
        percentile=98.0,
        window_size=100,
        min_samples=20,
    ):
        if not 0.0 < percentile < 100.0:
            raise ValueError("percentile must be between 0 and 100.")

        if window_size < 2:
            raise ValueError("window_size must be at least 2.")

        if min_samples < 2:
            raise ValueError("min_samples must be at least 2.")

        if min_samples > window_size:
            raise ValueError(
                "min_samples cannot be larger than window_size."
            )

        self.percentile = float(percentile)
        self.window_size = int(window_size)
        self.min_samples = int(min_samples)

        self.buf = deque(maxlen=self.window_size)

    def update(self, mse_value):
        """
        Add one nominal residual and update the operational radius.

        Returns
        -------
        tau : float
            Current operational radius. NaN until min_samples is reached.
        """
        self.buf.append(float(mse_value))

        if len(self.buf) < self.min_samples:
            return np.nan

        arr = np.asarray(self.buf, dtype=np.float64)

        return float(
            np.percentile(
                arr,
                self.percentile,
            )
        )

    def warmup_trace(self, mse_sequence):
        """
        Process an entire warm-up sequence and return tau(t).
        """
        history_tau = []

        for mse_value in mse_sequence:
            history_tau.append(
                self.update(mse_value)
            )

        return np.asarray(history_tau, dtype=np.float64)


# =====================================================================
# Convergence utilities
# =====================================================================

def operational_radius_convergence_trace(
    tau_history,
    *,
    lookback=10,
    eps=1e-12,
):
    """
    Relative variation of tau over a fixed look-back interval.

    C(t) = |tau(t) - tau(t-L)| / max(|tau(t-L)|, eps)
    """
    tau_history = np.asarray(tau_history, dtype=np.float64)

    convergence = np.full(
        tau_history.shape,
        np.nan,
        dtype=np.float64,
    )

    for t in range(lookback, len(tau_history)):
        current = tau_history[t]
        previous = tau_history[t - lookback]

        if np.isfinite(current) and np.isfinite(previous):
            convergence[t] = (
                abs(current - previous)
                / max(abs(previous), eps)
            )

    return convergence


def find_convergence_index(
    tau_history,
    *,
    lookback=10,
    tolerance=0.01,
    consecutive=10,
):
    """
    Return the first index at which the operational radius remains
    stable for `consecutive` updates.

    Returns None when convergence is not observed.
    """
    convergence = operational_radius_convergence_trace(
        tau_history,
        lookback=lookback,
    )

    stable_count = 0

    for t, value in enumerate(convergence):
        if np.isfinite(value) and value < tolerance:
            stable_count += 1

            if stable_count >= consecutive:
                return t
        else:
            stable_count = 0

    return None


# =====================================================================
# Shared simulator-trained AE applied unchanged to CWRU
# =====================================================================

def run_experiment(
    bearing_id,
    *,
    target_percentile=TARGET_PERCENTILE,
    warmup_size=100,
    operational_window_size=WINDOW_SIZE,
    min_warmup_samples=MIN_WARMUP_SAMPLES,
    convergence_lookback=10,
    convergence_tolerance=0.01,
    convergence_consecutive=10,
):
    """
    Evaluate the simulator-trained shared AE on one CWRU condition.

    Important
    ---------
    - `ae` and `scaler` are the models trained on synthetic healthy data.
    - They are transferred unchanged to CWRU.
    - All CWRU healthy windows are used only for the field-oracle p98.
    - Only the first `warmup_size` healthy windows are used by the
      online operational-radius estimator.
    """

    # -----------------------------------------------------------------
    # Load complete CWRU signals
    # -----------------------------------------------------------------

    normal_signal = load_cwru_signal(bearing_id, "healthy")
    ball_signal   = load_cwru_signal(bearing_id, "ball")
    inner_signal  = load_cwru_signal(bearing_id, "inner")
    outer_signal  = load_cwru_signal(bearing_id, "outer")

    print(f"\nCWRU operating condition: {bearing_id} RPM")
    print("Normal:", normal_signal.shape)
    print("Ball:  ", ball_signal.shape)
    print("Inner: ", inner_signal.shape)
    print("Outer: ", outer_signal.shape)

    # -----------------------------------------------------------------
    # Apply synthetic scaler and shared AE unchanged
    # -----------------------------------------------------------------

    def cwru_mse(signal):
        windows = make_windows(
            signal,
            win=SIG_LEN,
            step=STEP,
        )

        windows_2d = windows.reshape(
            windows.shape[0],
            -1,
        )

        # This scaler was fitted on synthetic nominal data.
        windows_scaled = scaler.transform(
            windows_2d
        )

        X = windows_scaled[
            :,
            :,
            np.newaxis,
        ].astype(np.float32)

        X_rec = ae.predict(
            X,
            verbose=0,
        )

        return np.mean(
            np.square(X - X_rec),
            axis=(1, 2),
        )

    # -----------------------------------------------------------------
    # Synthetic absolute threshold
    # -----------------------------------------------------------------

    X_train_rec = ae.predict(
        X_train,
        verbose=0,
    )

    mse_mc_healthy = np.mean(
        np.square(X_train - X_train_rec),
        axis=(1, 2),
    )

    tau_mc = float(
        np.percentile(
            mse_mc_healthy,
            target_percentile,
        )
    )

    # -----------------------------------------------------------------
    # CWRU residuals
    # -----------------------------------------------------------------

    mse_healthy = cwru_mse(normal_signal)
    mse_ball = cwru_mse(ball_signal)
    mse_inner = cwru_mse(inner_signal)
    mse_outer = cwru_mse(outer_signal)

    print(
        "Windows → "
        f"healthy:{len(mse_healthy)} "
        f"ball:{len(mse_ball)} "
        f"inner:{len(mse_inner)} "
        f"outer:{len(mse_outer)}"
    )

    # -----------------------------------------------------------------
    # Field oracle: all available healthy CWRU windows
    # -----------------------------------------------------------------

    tau_field_oracle = float(
        np.percentile(
            mse_healthy,
            target_percentile,
        )
    )

    # -----------------------------------------------------------------
    # Online field deployment: only initial healthy warm-up
    # -----------------------------------------------------------------

    if warmup_size is None:
        warmup_size = len(mse_healthy)

    warmup_size = min(
        int(warmup_size),
        len(mse_healthy),
    )

    if warmup_size < min_warmup_samples:
        raise ValueError(
            f"warmup_size={warmup_size} is smaller than "
            f"min_warmup_samples={min_warmup_samples}."
        )

    mse_warmup = mse_healthy[:warmup_size]

    adaptive = AdaptiveOperationalRadius(
        percentile=target_percentile,
        window_size=min(
            operational_window_size,
            warmup_size,
        ),
        min_samples=min(
            min_warmup_samples,
            warmup_size,
        ),
    )

    tau_history = adaptive.warmup_trace(
        mse_warmup
    )

    valid_tau = tau_history[
        np.isfinite(tau_history)
    ]

    if len(valid_tau) == 0:
        raise RuntimeError(
            "The online operational radius did not initialize."
        )

    tau_online = float(valid_tau[-1])

    convergence_index = find_convergence_index(
        tau_history,
        lookback=convergence_lookback,
        tolerance=convergence_tolerance,
        consecutive=convergence_consecutive,
    )

    # -----------------------------------------------------------------
    # Metrics
    # -----------------------------------------------------------------

    def alarm_rate(scores, threshold):
        return 100.0 * float(
            np.mean(scores > threshold)
        )

    thresholds = {
        "absolute_mc": tau_mc,
        "field_oracle": tau_field_oracle,
        "online_radius": tau_online,
    }

    rows = []

    for threshold_name, threshold_value in thresholds.items():
        rows.append(
            {
                "bearing_id": str(bearing_id),
                "detector": "Shared simulator-trained AE",
                "calibration": threshold_name,
                "threshold_percentile": target_percentile,
                "threshold": threshold_value,

                "healthy_far": alarm_rate(
                    mse_healthy,
                    threshold_value,
                ),
                "ball_dr": alarm_rate(
                    mse_ball,
                    threshold_value,
                ),
                "inner_race_dr": alarm_rate(
                    mse_inner,
                    threshold_value,
                ),
                "outer_race_dr": alarm_rate(
                    mse_outer,
                    threshold_value,
                ),

                "n_healthy": len(mse_healthy),
                "n_ball": len(mse_ball),
                "n_inner_race": len(mse_inner),
                "n_outer_race": len(mse_outer),

                "warmup_size": (
                    0
                    if threshold_name == "absolute_mc"
                    else (
                        len(mse_healthy)
                        if threshold_name == "field_oracle"
                        else warmup_size
                    )
                ),

                "convergence_index": (
                    convergence_index
                    if threshold_name == "online_radius"
                    else np.nan
                ),
            }
        )

    results_table = pd.DataFrame(rows)

    print("\nThresholds")
    print(f"τ_MC absolute       = {tau_mc:.6f}")
    print(f"τ_field oracle p98 = {tau_field_oracle:.6f}")
    print(f"τ_online           = {tau_online:.6f}")

    if convergence_index is None:
        print("Operational-radius convergence: not detected")
    else:
        print(
            "Operational-radius convergence detected at "
            f"warm-up window {convergence_index}."
        )

    display(results_table)

    return {
        "bearing_id": str(bearing_id),

        "mse_mc_healthy": mse_mc_healthy,
        "mse_healthy": mse_healthy,
        "mse_ball": mse_ball,
        "mse_inner": mse_inner,
        "mse_outer": mse_outer,

        "mse_warmup": mse_warmup,
        "tau_history": tau_history,

        "tau_mc": tau_mc,
        "tau_field_oracle": tau_field_oracle,
        "tau_online": tau_online,

        "convergence_index": convergence_index,
        "results_table": results_table,
    }


# =====================================================================
# Plotting
# =====================================================================

def plot_experiment(result):
    """
    Plot field residual distribution and online radius convergence.
    """

    bearing_id = result["bearing_id"]

    mse_healthy = result["mse_healthy"]
    mse_warmup = result["mse_warmup"]

    tau_mc = result["tau_mc"]
    tau_field_oracle = result["tau_field_oracle"]
    tau_online = result["tau_online"]

    tau_history = result["tau_history"]
    convergence_index = result["convergence_index"]

    # -----------------------------------------------------------------
    # Healthy residual distribution
    # -----------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(10, 5)
    )

    ax.hist(
        mse_healthy,
        bins=50,
        density=True,
        alpha=0.60,
        label="Healthy CWRU",
    )

    ax.axvline(
        tau_mc,
        linestyle="--",
        linewidth=1.5,
        label=f"Absolute simulator τ = {tau_mc:.4f}",
    )

    ax.axvline(
        tau_field_oracle,
        linestyle="--",
        linewidth=1.5,
        label=f"Field oracle p98 = {tau_field_oracle:.4f}",
    )

    ax.axvline(
        tau_online,
        linestyle="--",
        linewidth=1.5,
        label=f"Online radius = {tau_online:.4f}",
    )

    ax.set_xlabel("Reconstruction MSE")
    ax.set_ylabel("Density")
    ax.set_title(
        f"Healthy CWRU residual distribution — {bearing_id} RPM"
    )

    ax.legend()
    ax.grid(
        True,
        alpha=0.25,
        axis="y",
    )

    plt.tight_layout()
    plt.show()

    # -----------------------------------------------------------------
    # Operational-radius convergence
    # -----------------------------------------------------------------

    steps = np.arange(
        len(tau_history)
    )

    fig, ax = plt.subplots(
        figsize=(11, 5)
    )

    ax.plot(
        steps,
        tau_history,
        linewidth=1.5,
        label="Online operational radius",
    )

    ax.axhline(
        tau_field_oracle,
        linestyle="--",
        linewidth=1.2,
        label=f"Full-data field oracle = {tau_field_oracle:.4f}",
    )

    if convergence_index is not None:
        ax.axvline(
            convergence_index,
            linestyle=":",
            linewidth=1.2,
            label=f"Convergence = {convergence_index}",
        )

    ax.set_xlabel("Healthy warm-up window")
    ax.set_ylabel("Operational radius")
    ax.set_title(
        f"Operational-radius convergence — {bearing_id} RPM"
    )

    ax.legend()
    ax.grid(
        True,
        alpha=0.25,
    )

    plt.tight_layout()
    plt.show()

    print(
        f"Full healthy windows: {len(mse_healthy)}\n"
        f"Online warm-up windows: {len(mse_warmup)}\n"
        f"Field oracle p98: {tau_field_oracle:.6f}\n"
        f"Online radius: {tau_online:.6f}"
    )

In [21]:
# =====================================================================
# STABILITY UNDER LOCAL WARM-UP SAMPLING VARIABILITY
# Fixed OEM model; different 100-window field initialization samples
# =====================================================================

STABILITY_SEEDS = [7, 21, 42, 84, 126]

TARGET_PERCENTILE = 98.0
WARMUP_SIZE = 100
MIN_WARMUP_SAMPLES = 20

CONVERGENCE_LOOKBACK = 10
CONVERGENCE_TOLERANCE = 0.01
CONVERGENCE_CONSECUTIVE = 10

STEP = 1200

# ---------------------------------------------------------------------
# Sanity checks: the OEM artifacts must already exist
# ---------------------------------------------------------------------

required_objects = [
    "shared_ae",
    "shared_scaler",
    "shared_sig_len",
]

missing = [
    name for name in required_objects
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing fixed OEM artifacts: "
        + ", ".join(missing)
        + ". Run the shared-model training cell first."
    )


# ---------------------------------------------------------------------
# Helper: apply the fixed simulator-trained model to one field signal
# ---------------------------------------------------------------------

def shared_cwru_mse(signal):
    windows = make_windows(
        signal,
        win=shared_sig_len,
        step=STEP,
    ).astype(np.float32)

    windows_2d = windows.reshape(
        windows.shape[0],
        -1,
    )

    windows_scaled = shared_scaler.transform(
        windows_2d
    ).astype(np.float32)

    X = windows_scaled[..., np.newaxis]

    X_rec = shared_ae.predict(
        X,
        verbose=0,
    )

    return np.mean(
        np.square(X - X_rec),
        axis=(1, 2),
    )


# ---------------------------------------------------------------------
# Compute field residuals once.
# These do not change across stability seeds.
# ---------------------------------------------------------------------

field_residuals = {}

for bearing_id in BEARING_IDS:
    print(f"Computing fixed-model residuals for {bearing_id} RPM...")

    field_residuals[str(bearing_id)] = {
        "healthy": shared_cwru_mse(
            load_cwru_signal(bearing_id, "healthy")
        ),
        "ball": shared_cwru_mse(
            load_cwru_signal(bearing_id, "ball")
        ),
        "inner_race": shared_cwru_mse(
            load_cwru_signal(bearing_id, "inner")
        ),
        "outer_race": shared_cwru_mse(
            load_cwru_signal(bearing_id, "outer")
        ),
    }


# ---------------------------------------------------------------------
# Stability experiment
# ---------------------------------------------------------------------

warmup_stability_rows = []
warmup_traces = {}

for seed in STABILITY_SEEDS:

    print("\n" + "=" * 60)
    print(f"Warm-up sampling seed: {seed}")
    print("=" * 60)

    for bearing_id in BEARING_IDS:

        bearing_key = str(bearing_id)
        residuals = field_residuals[bearing_key]

        healthy_scores = residuals["healthy"]
        ball_scores = residuals["ball"]
        inner_scores = residuals["inner_race"]
        outer_scores = residuals["outer_race"]

        n_healthy = len(healthy_scores)

        if WARMUP_SIZE >= n_healthy:
            raise ValueError(
                f"{bearing_id}: WARMUP_SIZE={WARMUP_SIZE} must be "
                f"smaller than n_healthy={n_healthy} so that an "
                "independent healthy test set remains."
            )

        # Bearing-specific generator: reproducible but independent
        # across RPM conditions.
        rng = np.random.default_rng(
            seed + int(bearing_id)
        )

        # Random permutation determines both the selected observations
        # and the order in which they arrive during initialization.
        permutation = rng.permutation(n_healthy)

        warmup_indices = permutation[:WARMUP_SIZE]
        healthy_test_indices = permutation[WARMUP_SIZE:]

        warmup_scores = healthy_scores[warmup_indices]
        healthy_test_scores = healthy_scores[healthy_test_indices]

        # -------------------------------------------------------------
        # Sequential local-radius estimation
        # -------------------------------------------------------------

        adaptive = AdaptiveOperationalRadius(
            percentile=TARGET_PERCENTILE,
            window_size=WARMUP_SIZE,
            min_samples=MIN_WARMUP_SAMPLES,
        )

        tau_history = adaptive.warmup_trace(
            warmup_scores
        )

        valid_tau = tau_history[
            np.isfinite(tau_history)
        ]

        if len(valid_tau) == 0:
            raise RuntimeError(
                f"{bearing_id}, seed {seed}: "
                "the local radius did not initialize."
            )

        tau_online = float(valid_tau[-1])

        convergence_index = find_convergence_index(
            tau_history,
            lookback=CONVERGENCE_LOOKBACK,
            tolerance=CONVERGENCE_TOLERANCE,
            consecutive=CONVERGENCE_CONSECUTIVE,
        )

        # Complete-data field oracle, used only as a reference.
        tau_field_oracle = float(
            np.percentile(
                healthy_scores,
                TARGET_PERCENTILE,
            )
        )

        threshold_absolute_error = abs(
            tau_online - tau_field_oracle
        )

        threshold_relative_error_pct = (
            100.0
            * threshold_absolute_error
            / max(abs(tau_field_oracle), 1e-12)
        )

        def alarm_rate(scores):
            return 100.0 * float(
                np.mean(scores > tau_online)
            )

        row = {
            "bearing_id": bearing_key,
            "seed": seed,
            "detector": "Fixed shared simulator-trained AE",
            "calibration": "random_100_window_online_radius",
            "threshold_percentile": TARGET_PERCENTILE,

            "tau_online": tau_online,
            "tau_field_oracle": tau_field_oracle,
            "tau_absolute_error": threshold_absolute_error,
            "tau_relative_error_pct": threshold_relative_error_pct,

            # FAR is evaluated only on healthy windows not used
            # for local calibration.
            "healthy_far": alarm_rate(
                healthy_test_scores
            ),
            "ball_dr": alarm_rate(
                ball_scores
            ),
            "inner_race_dr": alarm_rate(
                inner_scores
            ),
            "outer_race_dr": alarm_rate(
                outer_scores
            ),

            "warmup_size": len(warmup_scores),
            "n_healthy_test": len(healthy_test_scores),
            "n_ball": len(ball_scores),
            "n_inner_race": len(inner_scores),
            "n_outer_race": len(outer_scores),

            # Convert zero-based Python index to window count.
            "convergence_window": (
                convergence_index + 1
                if convergence_index is not None
                else np.nan
            ),
        }

        warmup_stability_rows.append(row)

        warmup_traces[(bearing_key, seed)] = {
            "warmup_indices": warmup_indices,
            "healthy_test_indices": healthy_test_indices,
            "warmup_scores": warmup_scores,
            "tau_history": tau_history,
            "tau_online": tau_online,
            "tau_field_oracle": tau_field_oracle,
            "convergence_index": convergence_index,
        }

        print(
            f"{bearing_id} RPM | "
            f"tau={tau_online:.6f} | "
            f"oracle={tau_field_oracle:.6f} | "
            f"FAR={row['healthy_far']:.2f}% | "
            f"Ball={row['ball_dr']:.2f}% | "
            f"Conv={row['convergence_window']}"
        )


# ---------------------------------------------------------------------
# Long-form result table: one row per RPM and seed
# ---------------------------------------------------------------------

warmup_stability_results = pd.DataFrame(
    warmup_stability_rows
)

display(warmup_stability_results)


# ---------------------------------------------------------------------
# Summary by operating condition
# ---------------------------------------------------------------------

warmup_stability_by_bearing = (
    warmup_stability_results
    .groupby("bearing_id")
    .agg(
        tau_online_mean=("tau_online", "mean"),
        tau_online_std=("tau_online", "std"),

        tau_relative_error_mean=(
            "tau_relative_error_pct",
            "mean",
        ),
        tau_relative_error_std=(
            "tau_relative_error_pct",
            "std",
        ),

        healthy_far_mean=("healthy_far", "mean"),
        healthy_far_std=("healthy_far", "std"),

        ball_dr_mean=("ball_dr", "mean"),
        ball_dr_std=("ball_dr", "std"),

        inner_race_dr_mean=("inner_race_dr", "mean"),
        inner_race_dr_std=("inner_race_dr", "std"),

        outer_race_dr_mean=("outer_race_dr", "mean"),
        outer_race_dr_std=("outer_race_dr", "std"),

        convergence_mean=("convergence_window", "mean"),
        convergence_std=("convergence_window", "std"),
    )
    .reset_index()
)

display(warmup_stability_by_bearing)


# ---------------------------------------------------------------------
# Global stability across seeds
#
# First average the four RPM conditions within each seed. Then calculate
# variability across seeds, avoiding the conflation of RPM heterogeneity
# with warm-up sampling variability.
# ---------------------------------------------------------------------

warmup_seed_means = (
    warmup_stability_results
    .groupby("seed")[
        [
            "healthy_far",
            "ball_dr",
            "inner_race_dr",
            "outer_race_dr",
            "tau_relative_error_pct",
            "convergence_window",
        ]
    ]
    .mean()
)

display(warmup_seed_means)

warmup_global_summary = pd.DataFrame(
    {
        "mean": warmup_seed_means.mean(),
        "std": warmup_seed_means.std(ddof=1),
        "min": warmup_seed_means.min(),
        "max": warmup_seed_means.max(),
    }
)

display(warmup_global_summary)

Computing fixed-model residuals for 1730 RPM...
Computing fixed-model residuals for 1750 RPM...
Computing fixed-model residuals for 1772 RPM...
Computing fixed-model residuals for 1797 RPM...

Warm-up sampling seed: 7
1730 RPM | tau=0.293147 | oracle=0.294723 | FAR=2.63% | Ball=99.75% | Conv=39
1750 RPM | tau=0.314398 | oracle=0.312507 | FAR=0.33% | Ball=99.75% | Conv=47
1772 RPM | tau=0.308097 | oracle=0.307675 | FAR=1.65% | Ball=100.00% | Conv=39
1797 RPM | tau=0.337870 | oracle=0.346345 | FAR=4.85% | Ball=96.31% | Conv=45

Warm-up sampling seed: 21
1730 RPM | tau=0.294895 | oracle=0.294723 | FAR=1.97% | Ball=99.75% | Conv=69
1750 RPM | tau=0.297506 | oracle=0.312507 | FAR=7.24% | Ball=100.00% | Conv=58
1772 RPM | tau=0.322903 | oracle=0.307675 | FAR=0.00% | Ball=100.00% | Conv=66
1797 RPM | tau=0.346495 | oracle=0.346345 | FAR=1.94% | Ball=96.06% | Conv=77

Warm-up sampling seed: 42
1730 RPM | tau=0.286387 | oracle=0.294723 | FAR=6.25% | Ball=99.75% | Conv=39
1750 RPM | tau=0.309291

,bearing_id,seed,detector,calibration,threshold_percentile,tau_online,tau_field_oracle,tau_absolute_error,tau_relative_error_pct,healthy_far,ball_dr,inner_race_dr,outer_race_dr,warmup_size,n_healthy_test,n_ball,n_inner_race,n_outer_race,convergence_window
0,1730,7,Fixed shared simulator-trained AE,random_100_window_online_radius,98.0,0.293147,0.294723,0.001576,0.534731,2.631579,99.753086,100.0,100.000000,100,304,405,406,712,39
1,1750,7,Fixed shared simulator-trained AE,random_100_window_online_radius,98.0,0.314398,0.312507,0.001891,0.605008,0.328947,99.753086,100.0,100.000000,100,304,405,405,711,47
2,1772,7,Fixed shared simulator-trained AE,random_100_window_online_radius,98.0,0.308097,0.307675,0.000421,0.136882,1.650165,100.000000,100.0,100.000000,100,303,405,405,712,39
3,1797,7,Fixed shared simulator-trained AE,random_100_window_online_radius,98.0,0.337870,0.346345,0.008475,2.446991,4.854369,96.305419,100.0,100.000000,100,103,406,405,711,45
4,1730,21,Fixed shared simulator-trained AE,random_100_window_online_radius,98.0,0.294895,0.294723,0.000171,0.058106,1.973684,99.753086,100.0,100.000000,100,304,405,406,712,69
5,1750,21,Fixed shared simulator-trained AE,random_100_window_online_radius,98.0,0.297506,0.312507,0.015002,4.800483,7.236842,100.000000,100.0,100.000000,100,304,405,405,711,58
6,1772,21,Fixed shared simulator-trained AE,random_100_window_online_radius,98.0,0.322903,0.307675,0.015227,4.949195,0.000000,100.000000,100.0,99.859551,100,303,405,405,712,66
7,1797,21,Fixed shared simulator-trained AE,random_100_window_online_radius,98.0,0.346495,0.346345,0.000150,0.043212,1.941748,96.059113,100.0,100.000000,100,103,406,405,711,77
8,1730,42,Fixed shared simulator-trained AE,random_100_window_online_radius,98.0,0.286387,0.294723,0.008336,2.828391,6.250000,99.753086,100.0,100.000000,100,304,405,406,712,39
9,1750,42,Fixed shared simulator-trained AE,random_100_window_online_radius,98.0,0.309291,0.312507,0.003216,1.029066,3.289474,100.000000,100.0,100.000000,100,304,405,405,711,52


,bearing_id,tau_online_mean,tau_online_std,tau_relative_error_mean,tau_relative_error_std,healthy_far_mean,healthy_far_std,ball_dr_mean,ball_dr_std,inner_race_dr_mean,inner_race_dr_std,outer_race_dr_mean,outer_race_dr_std,convergence_mean,convergence_std
0,1730,0.290459,0.003967,1.470193,1.314218,4.078947,2.035756,99.753086,0.000000,100.0,0.0,100.00000,0.000000,45.6,13.145341
1,1750,0.308014,0.006309,1.679794,1.769776,3.486842,2.468202,99.950617,0.110423,100.0,0.0,100.00000,0.000000,54.2,13.553597
2,1772,0.309696,0.007595,1.442586,2.007818,2.178218,1.660036,100.000000,0.000000,100.0,0.0,99.97191,0.062811,49.4,13.831124
3,1797,0.341482,0.005148,1.578619,1.248317,3.495146,2.436874,96.157635,0.220302,100.0,0.0,100.00000,0.000000,56.2,15.006665


,healthy_far,ball_dr,inner_race_dr,outer_race_dr,tau_relative_error_pct,convergence_window
seed,,,,,,
7,2.366265,98.952898,100.0,100.000000,0.930903,42.50
21,2.788068,98.953050,100.0,99.964888,2.462749,67.50
42,2.797410,98.891474,100.0,100.000000,1.103158,48.25
84,4.341306,99.014626,100.0,100.000000,1.706792,52.50
126,4.255892,99.014626,100.0,100.000000,1.510390,46.00


,mean,std,min,max
healthy_far,3.309788,0.919796,2.366265,4.341306
ball_dr,98.965335,0.051528,98.891474,99.014626
inner_race_dr,100.000000,0.000000,100.000000,100.000000
outer_race_dr,99.992978,0.015703,99.964888,100.000000
tau_relative_error_pct,1.542798,0.600407,0.930903,2.462749
convergence_window,51.350000,9.730108,42.500000,67.500000


In [22]:
# =====================================================================
# EDGE FOOTPRINT MEASUREMENT
# Requires: shared_ae, shared_sig_len
# =====================================================================

import os
import time
import tempfile


# ---------------------------------------------------------------------
# 1. Parameter and raw-weight size
# ---------------------------------------------------------------------

parameter_count = shared_ae.count_params()

float32_weight_bytes = parameter_count * np.dtype(np.float32).itemsize
float32_weight_mb = float32_weight_bytes / 1_000_000
float32_weight_mib = float32_weight_bytes / (1024 ** 2)


# ---------------------------------------------------------------------
# 2. Exact serialized model size
# Save without optimizer state, because deployment uses inference only.
# ---------------------------------------------------------------------

with tempfile.TemporaryDirectory() as tmpdir:
    keras_path = os.path.join(tmpdir, "shared_oem_model.keras")

    # Clone/compile state is not required for inference.
    shared_ae.save(
        keras_path,
        include_optimizer=False,
    )

    serialized_bytes = os.path.getsize(keras_path)
    serialized_mb = serialized_bytes / 1_000_000
    serialized_mib = serialized_bytes / (1024 ** 2)


# ---------------------------------------------------------------------
# 3. Local calibration-state size
# Compact implementation: 100 float32 residuals plus a few scalar values.
# ---------------------------------------------------------------------

LOCAL_BUFFER_LENGTH = 100

residual_buffer_bytes = (
    LOCAL_BUFFER_LENGTH * np.dtype(np.float32).itemsize
)

# Example compact scalar state:
# current tau, previous tau, percentile, tolerance,
# convergence counter, state flag, and sample counter.
estimated_scalar_state_bytes = (
    4 * np.dtype(np.float32).itemsize
    + 3 * np.dtype(np.int32).itemsize
)

local_state_bytes = (
    residual_buffer_bytes + estimated_scalar_state_bytes
)


# ---------------------------------------------------------------------
# 4. Batch-1 CPU inference latency
# Use a real-shaped input. shared_ae is kept fixed.
# ---------------------------------------------------------------------

benchmark_input = np.zeros(
    (1, shared_sig_len, 1),
    dtype=np.float32,
)

benchmark_tensor = tf.convert_to_tensor(benchmark_input)

# Warm-up: TensorFlow tracing, memory allocation, and caches
N_WARMUP = 50
for _ in range(N_WARMUP):
    _ = shared_ae(
        benchmark_tensor,
        training=False,
    )

# Timed runs
N_RUNS = 500
latencies_ms = []

for _ in range(N_RUNS):
    start = time.perf_counter()

    output = shared_ae(
        benchmark_tensor,
        training=False,
    )

    # Force completion before stopping the clock
    _ = output.numpy()

    elapsed_ms = (
        time.perf_counter() - start
    ) * 1000.0

    latencies_ms.append(elapsed_ms)

latencies_ms = np.asarray(latencies_ms)

latency_median_ms = float(np.median(latencies_ms))
latency_mean_ms = float(np.mean(latencies_ms))
latency_std_ms = float(np.std(latencies_ms, ddof=1))
latency_p95_ms = float(np.percentile(latencies_ms, 95))


# ---------------------------------------------------------------------
# 5. Results
# ---------------------------------------------------------------------

edge_footprint = pd.DataFrame(
    [
        {
            "metric": "Model parameters",
            "value": f"{parameter_count:,}",
        },
        {
            "metric": "Raw float32 weight size",
            "value": (
                f"{float32_weight_mb:.2f} MB "
                f"({float32_weight_mib:.2f} MiB)"
            ),
        },
        {
            "metric": "Serialized .keras size",
            "value": (
                f"{serialized_mb:.2f} MB "
                f"({serialized_mib:.2f} MiB)"
            ),
        },
        {
            "metric": "CPU latency, median",
            "value": f"{latency_median_ms:.3f} ms/window",
        },
        {
            "metric": "CPU latency, mean ± std",
            "value": (
                f"{latency_mean_ms:.3f} ± "
                f"{latency_std_ms:.3f} ms/window"
            ),
        },
        {
            "metric": "CPU latency, p95",
            "value": f"{latency_p95_ms:.3f} ms/window",
        },
        {
            "metric": "Local residual-buffer length",
            "value": str(LOCAL_BUFFER_LENGTH),
        },
        {
            "metric": "Residual-buffer memory",
            "value": f"{residual_buffer_bytes} bytes",
        },
        {
            "metric": "Estimated compact local state",
            "value": (
                f"{local_state_bytes} bytes "
                f"(< 1 kB)"
            ),
        },
        {
            "metric": "Per-asset model training",
            "value": "None",
        },
        {
            "metric": "Fault labels required",
            "value": "None",
        },
    ]
)

display(edge_footprint)

,metric,value
0,Model parameters,"847,601"
1,Raw float32 weight size,3.39 MB (3.23 MiB)
2,Serialized .keras size,10.24 MB (9.76 MiB)
3,"CPU latency, median",6.261 ms/window
4,"CPU latency, mean ± std",6.239 ± 0.274 ms/window
5,"CPU latency, p95",6.617 ms/window
6,Local residual-buffer length,100
7,Residual-buffer memory,400 bytes
8,Estimated compact local state,428 bytes (< 1 kB)
9,Per-asset model training,None
